In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
DATA_PATH = "../data/synth_data_for_training.csv"
OUTPUT_DIR = "." # Save in current directory

# Set Plot Style
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 12})

def load_data():
    # Load headers first to avoid type warnings
    df_header = pd.read_csv(DATA_PATH, nrows=0)
    colnames = df_header.columns.tolist()
    
    # Load data
    df = pd.read_csv(DATA_PATH, skiprows=1, names=colnames, low_memory=False)
    
    # Clean Target
    df['checked'] = pd.to_numeric(df['checked'], errors='coerce')
    df = df.dropna(subset=['checked'])
    df['checked'] = df['checked'].astype(int)
    
    # Clean Features used for plotting
    cols_to_numeric = [
        'persoon_leeftijd_bij_onderzoek', 
        'adres_dagen_op_adres', 
        'relatie_overig_kostendeler'
    ]
    for col in cols_to_numeric:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
    return df

def plot_age_bias(df):
    """
    Generates Figure 1: Age Bias
    Claim: Under 25s are flagged 6.7x more often than over 55s.
    """
    print("\n--- Analysing Age Bias ---")
    
    # Create Age Bins
    def categorize_age(age):
        if age < 25: return "Young (<25)"
        elif age <= 55: return "Middle (25-55)"
        else: return "Senior (>55)"
        
    df['Age_Group'] = df['persoon_leeftijd_bij_onderzoek'].apply(categorize_age)
    
    # Order for the plot
    order = ["Young (<25)", "Middle (25-55)", "Senior (>55)"]
    
    # Calculate rates
    stats = df.groupby('Age_Group')['checked'].mean().reindex(order)
    print(stats)
    
    ratio = stats["Young (<25)"] / stats["Senior (>55)"]
    print(f"Ratio (Young / Senior): {ratio:.2f}x")

    # Plot
    plt.figure(figsize=(8, 5))
    ax = sns.barplot(x=stats.index, y=stats.values, palette="Blues_d")
    plt.title("Flag Rate by Age Group", fontsize=14, fontweight='bold')
    plt.ylabel("Proportion Flagged for Fraud")
    plt.xlabel("")
    plt.ylim(0, max(stats.values) * 1.2)
    
    # Add numbers on top
    for i, v in enumerate(stats.values):
        ax.text(i, v + 0.01, f"{v:.1%}", ha='center', fontweight='bold')
        
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/appendix_age_bias.png", dpi=300)
    print("Saved appendix_age_bias.png")

def plot_stability_bias(df):
    """
    Generates Figure 2: Stability Bias
    Claim: Recent movers (<1 year) flagged nearly 3x more often.
    """
    print("\n--- Analysing Stability Bias ---")
    
    def categorize_residence(days):
        if days < 365: return "New Resident\n(< 1 Year)"
        else: return "Long-Term\n(> 1 Year)"
        
    df['Residence_Type'] = df['adres_dagen_op_adres'].apply(categorize_residence)
    
    stats = df.groupby('Residence_Type')['checked'].mean()
    print(stats)
    
    # Note: Access using exact string keys from above
    rate_new = stats["New Resident\n(< 1 Year)"]
    rate_old = stats["Long-Term\n(> 1 Year)"]
    print(f"Ratio (New / Long-Term): {rate_new / rate_old:.2f}x")

    # Plot
    plt.figure(figsize=(6, 5))
    ax = sns.barplot(x=stats.index, y=stats.values, palette="Reds_d")
    plt.title("Flag Rate by Residence Duration", fontsize=14, fontweight='bold')
    plt.ylabel("Proportion Flagged")
    plt.xlabel("")
    plt.ylim(0, max(stats.values) * 1.2)
    
    for i, v in enumerate(stats.values):
        ax.text(i, v + 0.005, f"{v:.1%}", ha='center', fontweight='bold')
        
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/appendix_stability_bias.png", dpi=300)
    print("Saved appendix_stability_bias.png")

def plot_policy_bias(df):
    """
    Generates Figure 3: Policy/Cost Sharer Bias
    Claim: Housemates (Cost Sharers) flagged 2.3x more often.
    """
    print("\n--- Analysing Policy Bias (Cost Sharer) ---")
    
    df['Cost_Sharer_Label'] = df['relatie_overig_kostendeler'].map(
        {0: "Living Alone\n(No Cost Sharer)", 1: "Sharing Home\n(Cost Sharer)"}
    )
    
    stats = df.groupby('Cost_Sharer_Label')['checked'].mean()
    print(stats)
    
    rate_share = stats["Sharing Home\n(Cost Sharer)"]
    rate_alone = stats["Living Alone\n(No Cost Sharer)"]
    print(f"Ratio (Sharing / Alone): {rate_share / rate_alone:.2f}x")

    # Plot
    plt.figure(figsize=(6, 5))
    ax = sns.barplot(x=stats.index, y=stats.values, palette="Greens_d")
    plt.title("Flag Rate by Living Situation", fontsize=14, fontweight='bold')
    plt.ylabel("Proportion Flagged")
    plt.xlabel("")
    plt.ylim(0, max(stats.values) * 1.2)
    
    for i, v in enumerate(stats.values):
        ax.text(i, v + 0.005, f"{v:.1%}", ha='center', fontweight='bold')
        
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/appendix_policy_bias.png", dpi=300)
    print("Saved appendix_policy_bias.png")

if __name__ == "__main__":
    df = load_data()
    plot_age_bias(df)
    plot_stability_bias(df)
    plot_policy_bias(df)
    print("\nDone! All plots generated.")